# 01 — Exploratory data analysis

**Verity / Phase 2.** Dataset: `shivamb/real-or-fake-fake-jobposting-prediction`
(the Employment Scam Aegean Dataset, EMSCAD).

> **Kaggle setup (settings, not code edits):** turn **Internet = On** in the sidebar so
> `kagglehub` can download the dataset. No GPU needed for this notebook.

This notebook confronts the class-imbalance trap head-on: it prints the majority-class
baseline accuracy and states plainly why accuracy is never used as a headline metric.

In [ ]:
!pip -q install kagglehub

In [ ]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
SEED = 42

import kagglehub
DATA_DIR = kagglehub.dataset_download("shivamb/real-or-fake-fake-jobposting-prediction")
csvs = glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)
pref = [f for f in csvs if "fake_job_postings" in os.path.basename(f).lower()]
CSV = pref[0] if pref else csvs[0]
df = pd.read_csv(CSV)
print("Loaded:", CSV)
print("Shape:", df.shape)

In [ ]:
# --- Confirm the expected shape ---
n = len(df)
n_fraud = int(df["fraudulent"].sum())
prev = n_fraud / n
print(f"Rows:        {n:,}   (brief expects ~17,880)")
print(f"Fraudulent:  {n_fraud:,}   (brief expects ~866)")
print(f"Prevalence:  {prev:.4%}")
print(f"Columns ({df.shape[1]}): {list(df.columns)}")
print("Note: 18 columns include job_id and the fraudulent label -> 16 predictors.")

In [ ]:
# --- The imbalance trap, stated explicitly ---
maj_acc = (n - n_fraud) / n
print(f"MAJORITY-CLASS BASELINE (predict 'legitimate' for every posting):")
print(f"    accuracy      = {maj_acc:.4%}")
print(f"    fraud recall  = 0.0000   <-- catches ZERO scams")
print(f"    PR-AUC (no-skill) = prevalence = {prev:.4f}")
print()
print("This ~95% accuracy is exactly why ACCURACY IS NEVER REPORTED AS A HEADLINE")
print("METRIC in this project. A model can score 95% accuracy and be useless.")
print("We report PR-AUC (primary), F1, and recall@precision>=0.90 instead.")

In [ ]:
# --- Missing-value profile (NaN and blank strings both count as missing) ---
def missing_profile(frame):
    rows = []
    for c in frame.columns:
        na = int(frame[c].isna().sum())
        blank = int(frame[c].astype(str).str.strip().eq("").sum()) if frame[c].dtype == object else 0
        miss = na + blank
        rows.append((c, miss, round(100 * miss / len(frame), 1)))
    return pd.DataFrame(rows, columns=["column", "missing", "pct"]).sort_values("pct", ascending=False)

prof = missing_profile(df)
print(prof.to_string(index=False))
print()
print("salary_range is empty in ~84% of rows and has no currency or pay period -> see 03.")
print("required_education (~45%) and required_experience (~40%) are largely missing too.")

In [ ]:
# --- Class balance plot ---
import matplotlib.pyplot as plt
counts = df["fraudulent"].value_counts().sort_index()
plt.figure(figsize=(5, 4))
bars = plt.bar(["legitimate (0)", "fraudulent (1)"], counts.values,
               color=["#2ed3a0", "#f2545b"])
for i, v in enumerate(counts.values):
    plt.text(i, v, f"{v:,}\n{v/len(df):.1%}", ha="center", va="bottom", fontsize=10)
plt.title("EMSCAD class balance (4.8% fraudulent)")
plt.ylabel("count")
plt.tight_layout()
plt.savefig(os.path.join(OUT, "class_balance.png"), dpi=120)
plt.show()

### Takeaways

- The dataset is severely imbalanced (~4.8% fraudulent). Accuracy is meaningless here.
- Several structured fields are mostly missing; the served text model relies on text only.
- The reporting metrics for Phase 2 are **PR-AUC, F1, recall@precision≥0.90, and the full
  confusion matrix** — never headline accuracy.